# How Can Nonprofits Use AI to Personalize Something To Increase Donor Retention

In [1]:
import pandas as pd
import matplotlib.pyplot as pyplot
import numpy as np
import seaborn as sns
import scipy as sp

In [2]:
normalized_df = pd.read_csv('../ai_survey_normalized_clustering_data.csv')
survey_df = pd.read_csv('../ai_survey_results_2024_n=930.csv')

In [3]:
normalized_df.head()

,Unnamed: 0,nonprofit,org_years,regionality,org_small_med_large,global_north_south_int,collects_data,tech_person,merl_person,cloud_storage,...,[W] Generat,[W] Interpret,[W] Organi,[W] Predict,[W] Translat,[W] Other,[W] We don't know yet!,cluster3,cluster2,ai_want_2+
0,0,1.0,0.560000,0.25,1.0,0.0,1.0,0.0,0.0,1.0,...,0,0,1,1,0,0,0,1,1,1
1,1,1.0,0.046667,0.25,0.0,0.0,1.0,0.0,0.0,1.0,...,1,1,1,1,1,1,0,0,0,1
2,2,1.0,0.226667,0.25,0.0,0.0,1.0,1.0,0.0,1.0,...,0,1,0,0,1,0,0,-1,0,1
3,3,1.0,0.166667,0.50,0.0,0.0,1.0,1.0,1.0,1.0,...,0,1,1,1,0,0,0,1,1,1
4,4,1.0,0.053333,0.50,0.0,0.0,1.0,0.0,0.0,1.0,...,1,1,1,1,1,0,0,0,0,1


In [ ]:
survey_df.head()

,Unnamed: 0,source,Start time,nonprofit,non_org_type,role,person_org_years,org_size,org_years,continent,...,global_north_south_int,org_size_int,hubs_rural_urban_int,af_region_int,india_rural_urban_int,collab_feasibility_raw,person_ai_comfort_raw,org_years_raw,person_org_years_raw,cluster3
0,0,The big one,2024-03-07 11:03:50,1.0,NaN,Fundraiser,0.066667,31-60,0.560000,North America,...,0.0,5.0,NaN,NaN,NaN,7.0,7.0,84.0,4.0,1
1,1,The big one,2024-03-07 11:03:03,1.0,NaN,Leader,0.116667,0-5,0.046667,North America,...,0.0,0.0,NaN,NaN,NaN,4.0,5.0,7.0,7.0,0
2,2,The big one,2024-03-07 11:05:30,1.0,NaN,Leader,0.016667,6-15,0.226667,North America,...,0.0,1.0,NaN,NaN,NaN,3.0,10.0,34.0,1.0,-1
3,3,The big one,2024-03-07 11:05:30,1.0,NaN,Comms,0.133333,6-15,0.166667,North America,...,0.0,1.0,NaN,NaN,NaN,8.0,10.0,25.0,8.0,1
4,4,The big one,2024-03-07 11:07:06,1.0,NaN,Leader,0.133333,0-5,0.053333,North America,...,0.0,0.0,NaN,NaN,NaN,4.0,0.0,8.0,8.0,0


## Step 0: Data Cleaning

In [4]:
# Drop non_org_type (96% null) and strip whitespace across all string columns
survey_df = survey_df.drop(columns=['non_org_type'])

for col in survey_df.select_dtypes(include='object').columns:
    survey_df[col] = survey_df[col].str.strip()

/var/folders/wj/f5mwndv501bc3llhjvlrdcz80000gn/T/ipykernel_22111/1013878781.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in survey_df.select_dtypes(include='object').columns:


In [5]:
import ast

LIST_COLS = ['ai_use', 'ai_want', 'ai_risk', 'data_kinds']

def parse_list_col(val):
    if pd.isna(val):
        return []
    # replace non-breaking spaces before parsing
    val = val.replace('\xa0', ' ')
    try:
        items = ast.literal_eval(val)
        return [s.strip() for s in items if isinstance(s, str) and s.strip()]
    except Exception:
        return [val.strip()] if val.strip() else []

for col in LIST_COLS:
    if col in survey_df.columns:
        survey_df[col] = survey_df[col].apply(parse_list_col)

In [6]:
# Drop rows where org_opentext is too short to be usable (< 5 chars)
survey_df = survey_df[
    survey_df['org_opentext'].isna() | (survey_df['org_opentext'].str.len() >= 5)
].reset_index(drop=True)

In [7]:
# Impute person_ai_comfort_raw from normalized person_ai_comfort (scale: 0–10)
mask = survey_df['person_ai_comfort_raw'].isna() & survey_df['person_ai_comfort'].notna()
survey_df.loc[mask, 'person_ai_comfort_raw'] = (survey_df.loc[mask, 'person_ai_comfort'] * 10).round()

In [8]:
# Fill missing region values with 'Unknown'
survey_df['continent'] = survey_df['continent'].fillna('Unknown')
survey_df['global_north_south'] = survey_df['global_north_south'].fillna('Unknown')

In [9]:
# Merge survey and normalized datasets on index
merged_df = survey_df.merge(
    normalized_df,
    left_index=True,
    right_index=True,
    suffixes=('', '_norm')
)

print(f"survey_df: {survey_df.shape}, normalized_df: {normalized_df.shape}, merged_df: {merged_df.shape}")

survey_df: (906, 47), normalized_df: (930, 38), merged_df: (906, 85)
